In [0]:
%sql
CREATE VOLUME IF NOT EXISTS whriv.economics.reference;

In [0]:
import requests, time, re
import pandas as pd

FRED_KEY = dbutils.secrets.get(scope="fred", key="api_key")

def fred_get(endpoint, **params):
    r = requests.get(
        f"https://api.stlouisfed.org/fred/{endpoint}",
        params={**params, "api_key": FRED_KEY, "file_type": "json"},
        timeout=30,
    )
    data = r.json()
    if r.status_code != 200:
        raise Exception(data.get("error_message", f"HTTP {r.status_code}"))
    return data

In [0]:
CSV_PATH = "/Volumes/whriv/economics/reference/Sheet 1-cpi_relative_importance.csv"  # adjust to your file name
components = pd.read_csv(CSV_PATH, dtype=str).fillna("")
print(len(components), "rows")

# every ID should be CUUR0000 + item_code
expected = "CUUR0000" + components["item_code"].str.strip()
mismatch = components[components["fred_series_id_nsa"].str.strip() != expected]
print("rows where the ID doesn't match CUUR0000 + item_code:", len(mismatch))

In [0]:
def norm(s):
    return re.sub(r"[^a-z0-9 ]", "", s.lower().replace("&", "and")).strip()

def core_title(t):
    t = t.replace("Consumer Price Index for All Urban Consumers: ", "")
    return re.sub(r"\s+in U\.S\. City Average$", "", t)

results = []
for _, r in components.iterrows():
    sid = r["fred_series_id_nsa"].strip()
    row = {"item_name": r["item_name"], "series_id": sid}
    try:
        m = fred_get("series", series_id=sid)["seriess"][0]
        title_ok = norm(core_title(m["title"])) == norm(r["item_name"])
        format_ok = (m["seasonal_adjustment_short"] == "NSA"
                     and m["frequency_short"] == "M"
                     and m["observation_start"] <= "2018-01-01")
        row.update(status="OK" if (title_ok and format_ok) else "REVIEW",
                   seasonal=m["seasonal_adjustment_short"], freq=m["frequency_short"],
                   start=m["observation_start"], end=m["observation_end"],
                   fred_title=m["title"])
    except Exception as e:
        row.update(status="MISSING", fred_title=str(e))
    results.append(row)
    time.sleep(0.3)

res = pd.DataFrame(results)
display(res)
print(res["status"].value_counts())